# 📓 Model Evaluation and Improvement

## 🧱 Introduction

In this notebook, we’ll walk through the complete process of evaluating and improving machine learning models.
We’ll use the Titanic dataset to:
- Preprocess data
- Train 3 classifiers (Logistic Regression, Decision Tree, KNN)
- Evaluate using cross-validation
- Tune hyperparameters
- Select the best model based on metrics and interpretation

---

## Import the necessary Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import cross_val_score, cross_validate, StratifiedKFold, GridSearchCV, RandomizedSearchCV

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

## 🚢 Load Dataset

In [2]:
# Load Titanic dataset
X, y = fetch_openml(data_id=40945, as_frame=True, return_X_y=True)
X['survived'] = y.astype(int)
y = X['survived']
X = X.drop(columns='survived')

In [3]:
X.head()

,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [4]:
y

0       1
1       1
2       0
3       0
4       0
       ..
1304    0
1305    0
1306    0
1307    0
1308    0
Name: survived, Length: 1309, dtype: int32

---
## 🔍 Preprocessing

We define preprocessing steps using `ColumnTransformer` and `Pipeline`. We impute missing values, scale numeric features, and encode categoricals.
This ensures consistency and keeps transformations aligned with each model.

In [5]:
numeric_features = ['age', 'fare']
categorical_features = ['sex', 'pclass', 'embarked', 'sibsp', 'parch']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

---
## ✅ Task 1: Set Up Models

In [6]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier()
}

# Create Stratified K-Fold (5 splits)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

---
## 🔁 Task 2: Cross-Validation

In [7]:
scoring = ['accuracy', 'precision', 'recall', 'f1']
results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('clf', model)
    ])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring=scoring)
    results[name] = {metric: np.mean(scores[f'test_{metric}']) for metric in scoring}

pd.DataFrame(results).T  # Model comparison table

,accuracy,precision,recall,f1
Logistic Regression,0.795277,0.744484,0.706,0.724153
Decision Tree,0.760870,0.696667,0.666,0.679799
KNN,0.796028,0.760444,0.684,0.718678


- Logistic Regression often performs well with good recall and precision.
- KNN may suffer due to feature scaling sensitivity.
- Decision Tree may slightly overfit but can be competitive.

Interpretation:
- Logistic Regression had solid overall performance with balanced metrics.
- KNN had slightly better precision but slightly lower recall.
- Decision Tree underperformed slightly on all metrics, showing possible overfitting.

---
## ⚙️ Task 3: Hyperparameter Tuning

In [8]:
param_grids = {
    'Logistic Regression': {
        'clf__C': [0.01, 0.1, 1, 10]
    },
    'Decision Tree': {
        'clf__max_depth': [3, 5, 7, None],
        'clf__min_samples_split': [2, 4, 6]
    },
    'KNN': {
        'clf__n_neighbors': [3, 5, 7, 9]
    }
}

best_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('clf', model)
    ])
    search = GridSearchCV(pipeline, param_grids[name], cv=cv, scoring='f1')
    search.fit(X, y)
    best_models[name] = {
        'best_estimator': search.best_estimator_,
        'best_params': search.best_params_,
        'best_score': search.best_score_
    }

# Visualize best parameters
best_params_df = pd.DataFrame({name: data['best_params'] for name, data in best_models.items()})
best_params_df.T

,clf__C,clf__max_depth,clf__min_samples_split,clf__n_neighbors
Logistic Regression,1.0,NaN,NaN,NaN
Decision Tree,NaN,3.0,2.0,NaN
KNN,NaN,NaN,NaN,7.0


- **Logistic Regression** performed best with `C=1.0`, indicating moderate regularization.
- **Decision Tree** achieved its highest F1-score with `max_depth=3` and `min_samples_split=2`, which likely reduced overfitting while preserving important splits.
- **KNN** performed best with `n_neighbors=7`, balancing over-sensitivity (low k) and over-smoothing (high k).
These optimal parameters reflect the importance of controlled complexity in models applied to real-world data.

---
## 🧠 Task 4: Final Model Selection

In [9]:
final_results = {
    name: model_info['best_score']
    for name, model_info in best_models.items()
}
pd.Series(final_results).sort_values(ascending=False)

Decision Tree          0.729452
KNN                    0.727330
Logistic Regression    0.724153
dtype: float64

- The **Decision Tree** model performed best after tuning, F1-score = 0.729.
- **KNN** also performed well, showing its potential when tuned properly.
- **Logistic Regression** remained consistent but was slightly outperformed.

---
## 📝 Final Summary

Summary:
- We evaluated 3 models using Stratified K-Fold and multiple metrics
- We tuned hyperparameters with GridSearchCV
- We compared results fairly and selected the best model based on F1-score and interpretability

Lessons:
- Evaluation across metrics provides a more complete view
- Tuning can significantly improve models
- Model choice should be based on performance and real-world considerations

---
## ✅ Conclusion

You've now experienced the full process of evaluating and improving machine learning models:
•	You applied consistent cross-validation to ensure reliable performance estimates.
•	You tuned model hyperparameters to boost effectiveness.
•	You made evidence-based decisions to select the most appropriate model.
These practices are essential to creating models you can trust. Keep iterating, testing, and refining — this is the mindset of a real data scientist!

🎉 Well done! You've practiced reliable model evaluation, tuning, and selection — key steps in real-world data science. Keep practicing, stay curious, and trust your data-informed models!

---